# Brain Tumor MRI Classification with ResNet50 and GWN

This notebook contains the main experiment from:

**Beyond Accuracy: Comparison of ResNet50 and GWN-Enhanced Models for Brain Tumor MRI Classification with LIME Visualization**

The notebook compares an ImageNet-pretrained ResNet50 with a ResNet50 + GWN model on three classes: No Tumor, Glioma, and Pituitary.

## Experimental setup

- 1,100 images per class
- 80/10/10 train, validation, and test split
- Input size: 224 × 224
- Random horizontal flip and ±15° rotation for training
- Adam optimizer
- 5 warm-up epochs at `1e-4`
- Fine-tuning up to 30 epochs at `5e-5`
- ReduceLROnPlateau and early stopping
- LIME for visual explanation

I keep the image-level split used in the paper because the public dataset does not include patient IDs.

## Dataset

Expected folder structure:

```text
Training/
├── notumor/
├── glioma/
└── pituitary/
```

Change `DATA_DIR` below if your dataset is stored somewhere else.


In [ ]:
# Uncomment if the packages are not installed
# !pip install -q lime scikit-image

import os
import random
import copy
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.preprocessing import label_binarize

from lime import lime_image
from skimage.segmentation import mark_boundaries
from tqdm.auto import tqdm


DATA_DIR = Path("./Training")

SEED = 42
N_PER_CLASS = 1100
IMG_SIZE = 224
BATCH_SIZE = 32

PHASE1_EPOCHS = 5
PHASE2_EPOCHS = 30
PATIENCE = 3

LR_PHASE1 = 1e-4
LR_PHASE2 = 5e-5
WEIGHT_DECAY = 1e-4

GWN_GROUPS = 2
DROPOUT = 0.5

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Expected folders: glioma, notumor, pituitary
CLASS_NAMES = ["notumor", "glioma", "pituitary"]
extensions = {".jpg", ".jpeg", ".png", ".bmp"}

rows = []
for label, class_name in enumerate(CLASS_NAMES):
    class_dir = DATA_DIR / class_name
    paths = sorted(p for p in class_dir.iterdir() if p.suffix.lower() in extensions)

    if len(paths) < N_PER_CLASS:
        raise ValueError(f"{class_name}: found {len(paths)} images, need {N_PER_CLASS}")

    rng = random.Random(SEED)
    selected = rng.sample(paths, N_PER_CLASS)

    rows.extend(
        {"path": str(path), "label": label, "class_name": class_name}
        for path in selected
    )

df = pd.DataFrame(rows)

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label"],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SEED
)

for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    counts = split["class_name"].value_counts().reindex(CLASS_NAMES)
    print(name, len(split), counts.to_dict())


In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class MRIDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        return self.transform(image), int(row["label"])

train_ds = MRIDataset(train_df, train_tf)
val_ds = MRIDataset(val_df, eval_tf)
test_ds = MRIDataset(test_df, eval_tf)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=torch.cuda.is_available()
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=torch.cuda.is_available()
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=torch.cuda.is_available()
)


In [ ]:
class GWNConv2d(nn.Conv2d):
    def __init__(self, *args, ghost_groups=2, eps=1e-6, **kwargs):
        super().__init__(*args, **kwargs)
        self.ghost_groups = max(1, ghost_groups)
        self.eps = eps

    def forward(self, x):
        w = self.weight
        out_channels = w.shape[0]
        groups = min(self.ghost_groups, out_channels)

        pad = (-out_channels) % groups
        if pad:
            w = F.pad(w, (0, 0, 0, 0, 0, 0, 0, pad))

        shape = w.shape
        w = w.reshape(groups, shape[0] // groups, -1)
        w = w / torch.sqrt((w ** 2).sum(dim=-1, keepdim=True) + self.eps)
        w = w.reshape(shape)[:out_channels]

        return F.conv2d(
            x, w, self.bias, self.stride,
            self.padding, self.dilation, self.groups
        )


class GWNLinear(nn.Linear):
    def __init__(self, *args, ghost_groups=2, eps=1e-6, **kwargs):
        super().__init__(*args, **kwargs)
        self.ghost_groups = max(1, ghost_groups)
        self.eps = eps

    def forward(self, x):
        w = self.weight
        out_features = w.shape[0]
        groups = min(self.ghost_groups, out_features)

        pad = (-out_features) % groups
        if pad:
            w = F.pad(w, (0, 0, 0, pad))

        shape = w.shape
        w = w.reshape(groups, shape[0] // groups, -1)
        w = w / torch.sqrt((w ** 2).sum(dim=-1, keepdim=True) + self.eps)
        w = w.reshape(shape)[:out_features]

        return F.linear(x, w, self.bias)


def apply_gwn(module, ghost_groups=2):
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Conv2d):
            new = GWNConv2d(
                child.in_channels,
                child.out_channels,
                child.kernel_size,
                stride=child.stride,
                padding=child.padding,
                dilation=child.dilation,
                groups=child.groups,
                bias=child.bias is not None,
                padding_mode=child.padding_mode,
                ghost_groups=ghost_groups,
            )
            new.load_state_dict(child.state_dict())
            setattr(module, name, new)

        elif isinstance(child, nn.Linear):
            new = GWNLinear(
                child.in_features,
                child.out_features,
                bias=child.bias is not None,
                ghost_groups=ghost_groups,
            )
            new.load_state_dict(child.state_dict())
            setattr(module, name, new)

        else:
            apply_gwn(child, ghost_groups)

    return module


In [ ]:
def build_model(use_gwn=False):
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    in_features = model.fc.in_features

    model.fc = nn.Sequential(
        nn.Dropout(DROPOUT),
        nn.Linear(in_features, len(CLASS_NAMES))
    )

    if use_gwn:
        model = apply_gwn(model, GWN_GROUPS)

    return model


In [ ]:
criterion = nn.CrossEntropyLoss()

def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total = 0

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for images, labels in tqdm(loader, leave=False):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            if is_train:
                optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, labels)

            if is_train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * labels.size(0)
            total += labels.size(0)

    return total_loss / total


In [ ]:
# ResNet50 baseline

baseline = build_model(use_gwn=False).to(DEVICE)

# warm-up: train only the classifier
for p in baseline.parameters():
    p.requires_grad = False
for p in baseline.fc.parameters():
    p.requires_grad = True

optimizer = torch.optim.Adam(
    baseline.fc.parameters(),
    lr=LR_PHASE1,
    weight_decay=WEIGHT_DECAY
)

hist_baseline = {"train_loss": [], "val_loss": []}

for epoch in range(PHASE1_EPOCHS):
    train_loss = run_epoch(baseline, train_loader, optimizer)
    val_loss = run_epoch(baseline, val_loader)

    hist_baseline["train_loss"].append(train_loss)
    hist_baseline["val_loss"].append(val_loss)

    print(
        f"warm-up {epoch + 1}/{PHASE1_EPOCHS} | "
        f"train {train_loss:.4f} | val {val_loss:.4f}"
    )

# fine-tune all layers
for p in baseline.parameters():
    p.requires_grad = True

optimizer = torch.optim.Adam(
    baseline.parameters(),
    lr=LR_PHASE2,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)

best_val = float("inf")
best_state = None
wait = 0
epoch_baseline = 0

for epoch in range(PHASE2_EPOCHS):
    train_loss = run_epoch(baseline, train_loader, optimizer)
    val_loss = run_epoch(baseline, val_loader)
    scheduler.step(val_loss)

    hist_baseline["train_loss"].append(train_loss)
    hist_baseline["val_loss"].append(val_loss)

    print(
        f"fine-tune {epoch + 1}/{PHASE2_EPOCHS} | "
        f"train {train_loss:.4f} | val {val_loss:.4f}"
    )

    if val_loss < best_val:
        best_val = val_loss
        best_state = copy.deepcopy(baseline.state_dict())
        epoch_baseline = epoch + 1
        wait = 0
    else:
        wait += 1

    if wait >= PATIENCE:
        print("early stopping")
        break

baseline.load_state_dict(best_state)
torch.save(best_state, "resnet50_best.pth")


In [ ]:
# ResNet50 + GWN

gwn_model = build_model(use_gwn=True).to(DEVICE)

for p in gwn_model.parameters():
    p.requires_grad = False
for p in gwn_model.fc.parameters():
    p.requires_grad = True

optimizer = torch.optim.Adam(
    gwn_model.fc.parameters(),
    lr=LR_PHASE1,
    weight_decay=WEIGHT_DECAY
)

hist_gwn = {"train_loss": [], "val_loss": []}

for epoch in range(PHASE1_EPOCHS):
    train_loss = run_epoch(gwn_model, train_loader, optimizer)
    val_loss = run_epoch(gwn_model, val_loader)

    hist_gwn["train_loss"].append(train_loss)
    hist_gwn["val_loss"].append(val_loss)

    print(
        f"warm-up {epoch + 1}/{PHASE1_EPOCHS} | "
        f"train {train_loss:.4f} | val {val_loss:.4f}"
    )

for p in gwn_model.parameters():
    p.requires_grad = True

optimizer = torch.optim.Adam(
    gwn_model.parameters(),
    lr=LR_PHASE2,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)

best_val = float("inf")
best_state = None
wait = 0
epoch_gwn = 0

for epoch in range(PHASE2_EPOCHS):
    train_loss = run_epoch(gwn_model, train_loader, optimizer)
    val_loss = run_epoch(gwn_model, val_loader)
    scheduler.step(val_loss)

    hist_gwn["train_loss"].append(train_loss)
    hist_gwn["val_loss"].append(val_loss)

    print(
        f"fine-tune {epoch + 1}/{PHASE2_EPOCHS} | "
        f"train {train_loss:.4f} | val {val_loss:.4f}"
    )

    if val_loss < best_val:
        best_val = val_loss
        best_state = copy.deepcopy(gwn_model.state_dict())
        epoch_gwn = epoch + 1
        wait = 0
    else:
        wait += 1

    if wait >= PATIENCE:
        print("early stopping")
        break

gwn_model.load_state_dict(best_state)
torch.save(best_state, "resnet50_gwn_best.pth")

print("Best fine-tuning epoch")
print("ResNet50:", epoch_baseline)
print("ResNet50 + GWN:", epoch_gwn)


In [ ]:
def predict(model, loader):
    model.eval()
    y_true, y_pred, y_prob = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            logits = model(images.to(DEVICE))
            prob = F.softmax(logits, dim=1).cpu().numpy()

            y_true.extend(labels.numpy())
            y_pred.extend(prob.argmax(axis=1))
            y_prob.extend(prob)

    return (
        np.asarray(y_true),
        np.asarray(y_pred),
        np.asarray(y_prob)
    )


def evaluate(model):
    y_true, y_pred, y_prob = predict(model, test_loader)
    y_bin = label_binarize(y_true, classes=np.arange(len(CLASS_NAMES)))
    cm = confusion_matrix(y_true, y_pred)

    sensitivity = np.diag(cm) / cm.sum(axis=1)

    specificity = []
    for i in range(len(CLASS_NAMES)):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        specificity.append(tn / (tn + fp))

    per_class_auc = roc_auc_score(
        y_bin, y_prob, average=None, multi_class="ovr"
    )

    summary = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_auc": roc_auc_score(
            y_bin, y_prob, average="macro", multi_class="ovr"
        ),
        "macro_f1": f1_score(y_true, y_pred, average="macro")
    }

    per_class = pd.DataFrame({
        "class": CLASS_NAMES,
        "AUC": per_class_auc,
        "sensitivity": sensitivity,
        "specificity": specificity
    })

    return summary, per_class, cm


base_summary, base_class, base_cm = evaluate(baseline)
gwn_summary, gwn_class, gwn_cm = evaluate(gwn_model)

display(pd.DataFrame(
    [base_summary, gwn_summary],
    index=["ResNet50", "ResNet50 + GWN"]
))

print("\nResNet50")
display(base_class)

print("\nResNet50 + GWN")
display(gwn_class)


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(hist_baseline["train_loss"], label="ResNet50 train")
plt.plot(hist_baseline["val_loss"], label="ResNet50 val")
plt.plot(hist_gwn["train_loss"], label="GWN train")
plt.plot(hist_gwn["val_loss"], label="GWN val")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
lime_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

def lime_predict(model, images):
    batch = []
    for image in images:
        image = np.clip(image, 0, 1)
        pil = Image.fromarray((image * 255).astype(np.uint8))
        batch.append(lime_tf(pil))

    batch = torch.stack(batch).to(DEVICE)

    with torch.no_grad():
        return F.softmax(model(batch), dim=1).cpu().numpy()


def load_lime_image(idx):
    row = test_df.reset_index(drop=True).iloc[idx]
    image = Image.open(row["path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    return np.asarray(image, dtype=np.float32) / 255.0, int(row["label"])


explainer = lime_image.LimeImageExplainer(random_state=SEED)
rng = np.random.default_rng(SEED)
sample_indices = rng.choice(len(test_df), size=5, replace=False)

for idx in sample_indices:
    image, true_label = load_lime_image(idx)

    exp_base = explainer.explain_instance(
        image,
        classifier_fn=lambda x: lime_predict(baseline, x),
        top_labels=1,
        num_samples=1000,
        random_seed=SEED
    )

    exp_gwn = explainer.explain_instance(
        image,
        classifier_fn=lambda x: lime_predict(gwn_model, x),
        top_labels=1,
        num_samples=1000,
        random_seed=SEED
    )

    pred_base = exp_base.top_labels[0]
    pred_gwn = exp_gwn.top_labels[0]

    vis_base, mask_base = exp_base.get_image_and_mask(
        pred_base,
        positive_only=True,
        num_features=5,
        hide_rest=True
    )

    vis_gwn, mask_gwn = exp_gwn.get_image_and_mask(
        pred_gwn,
        positive_only=True,
        num_features=5,
        hide_rest=True
    )

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(image)
    axes[0].set_title(f"True: {CLASS_NAMES[true_label]}")

    axes[1].imshow(mark_boundaries(vis_base, mask_base))
    axes[1].set_title(f"ResNet50: {CLASS_NAMES[pred_base]}")

    axes[2].imshow(mark_boundaries(vis_gwn, mask_gwn))
    axes[2].set_title(f"ResNet50 + GWN: {CLASS_NAMES[pred_gwn]}")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()
